# v21b terrain campaign — training progress

Reads rsl-rl stdout logs (`train_<run>.log`) and renders the campaign telemetry:
terrain-curriculum level, velocity-progress income, the gate-metric trap,
exploration (action std), velocity errors vs the flat baseline, and deaths.

**Data**: point `RUNS` at local log copies. To fetch fresh logs from the GPU box:

```bash
rsync dvv-gpu:~/dev/bots/luwu_mjlab/train_v21b_v{4,5,6}.log luwu_mjlab/
```

Context for the run labels (2026-07-16 campaign):
- **v4** — warm-started, entropy 0.001; converged to *standing* (the reward-dominant
  optimum before the progress rewards existed).
- **v5** — + velocity-progress rewards and terrain-level-scaled hazard DR; walked, but
  the frac-gated terrain curriculum demoted every walker to row 0 (killed at 1.1k).
- **v6** — + ratio-gated terrain promotion (promote ≥ 0.5, demote < 0.25); the first
  run that walks AND earns terrain promotion.


In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np

ROOT = Path("..")  # notebook lives in notebooks/; logs sit at the repo root

# label -> log path (missing files are skipped with a note)
RUNS = {
    "v4": ROOT / "train_v21b_v4.log",
    "v5": ROOT / "train_v21b_v5.log",
    "v6": ROOT / "train_v21b_v6.log",
}
# Flat-walking reference for the error chart (v21a extension run); optional.
BASELINE_LOG = ROOT / "train_v21a_v1ext.log"

# metric label -> unique substring of its log line (value = last token)
METRICS = {
    "terrain_levels": "Curriculum/terrain_levels:",
    "progress_lin": "Episode_Reward/velocity_progress_lin:",
    "progress_ang": "Episode_Reward/velocity_progress_ang:",
    "err_vel_xy": "Metrics/twist/error_vel_xy:",
    "err_vel_yaw": "Metrics/twist/error_vel_yaw:",
    "eplen": "Mean episode length:",
    "reward": "Mean reward:",
    "std": "Mean action std:",
    "seed_tracking": "Curriculum/command_grid/seed_tracking:",
    "track_lin": "Episode_Reward/track_linear_velocity:",
    "term_illegal": "Episode_Termination/illegal_contact:",
    "term_fell": "Episode_Termination/fell_over:",
}

STRIDE = 25  # keep every Nth iteration sample


In [ ]:
def parse_log(path, metrics=METRICS, stride=STRIDE):
    """One value per training iteration per metric, in file order.

    rsl-rl prints each metric once per iteration block, so the n-th
    occurrence of a metric line belongs to iteration n. Non-numeric
    tails (the config-dump table at the top of the log) are skipped.
    """
    series = {name: [] for name in metrics}
    with open(path, errors="replace") as fh:
        for line in fh:
            for name, needle in metrics.items():
                if needle in line:
                    tail = line.rsplit(":", 1)[-1].strip()
                    try:
                        series[name].append(float(tail))
                    except ValueError:
                        pass
                    break
    out = {}
    for name, vals in series.items():
        arr = np.asarray(vals)
        out[name] = (np.arange(len(arr))[::stride], arr[::stride])
    return out


runs = {}
for label, path in RUNS.items():
    if path.exists():
        runs[label] = parse_log(path)
        n = len(runs[label]["reward"][1]) * STRIDE
        print(f"{label}: {path.name}, ~{n} iterations")
    else:
        print(f"{label}: {path} missing — skipped")

baseline = {}
if BASELINE_LOG.exists():
    b = parse_log(BASELINE_LOG, metrics={
        "err_vel_xy": METRICS["err_vel_xy"],
        "err_vel_yaw": METRICS["err_vel_yaw"],
    })
    for k, (it, v) in b.items():
        steady = v[it >= 1000]
        if len(steady):
            baseline[k] = float(steady.mean())
    print("flat baseline (v21a steady state):", {k: round(v, 3) for k, v in baseline.items()})


In [ ]:
# Reference dataviz palette (light surface), one fixed hue per run entity.
RUN_COLOR = {"v4": "#2a78d6", "v5": "#008300", "v6": "#e87ba4"}
MET_COLOR = ("#eb6834", "#4a3aa7")  # in-run metric pairs (orange / violet)
INK, INK2, MUTED, GRID = "#0b0b0b", "#52514e", "#898781", "#e1e0d9"

plt.rcParams.update({
    "figure.facecolor": "#fcfcfb", "axes.facecolor": "#fcfcfb",
    "axes.edgecolor": "#c3c2b7", "axes.labelcolor": INK2,
    "axes.grid": True, "grid.color": GRID, "grid.linewidth": 0.8,
    "axes.spines.top": False, "axes.spines.right": False,
    "xtick.color": MUTED, "ytick.color": MUTED,
    "font.size": 10, "figure.dpi": 110,
})


def draw(ax, series, title, ylabel, end_fmt="{:.2f}"):
    """series: list of (iters, values, label, color)."""
    for it, v, label, color in series:
        if not len(v):
            continue
        ax.plot(it, v, color=color, lw=1.8, label=label, solid_joinstyle="round")
        ax.scatter([it[-1]], [v[-1]], s=14, color=color, zorder=3)
        ax.annotate(f"{label} {end_fmt.format(v[-1])}", (it[-1], v[-1]),
                    xytext=(6, 0), textcoords="offset points",
                    fontsize=8.5, fontweight="bold", color=INK2, va="center")
    ax.set_title(title, fontsize=11, fontweight=600, color=INK, loc="left")
    ax.set_xlabel("iteration")
    ax.set_ylabel(ylabel)
    ax.margins(x=0.11)
    ax.legend(loc="best", fontsize=8.5, frameon=False)


def run_series(metric):
    return [(*runs[r][metric], r, RUN_COLOR[r]) for r in runs if metric in runs[r]]


In [ ]:
fig, ax = plt.subplots(figsize=(9.5, 4.2))
draw(ax, run_series("terrain_levels"),
     "Terrain curriculum — mean difficulty level (rows 0-9)", "mean level")
ax.text(0.99, 0.97,
        "v4 climbed by STANDING (hollow gate passes)\n"
        "v5 walked, frac gates demoted all walkers to row 0\n"
        "v6 walks, ratio gates -> earned monotone climb",
        transform=ax.transAxes, ha="right", va="top", fontsize=8.5, color=INK2)
plt.show()


In [ ]:
fig, ax = plt.subplots(figsize=(9.5, 3.8))
draw(ax, [s for s in run_series("seed_tracking") if s[2] in ("v4", "v6")],
     "The gate-metric trap — seed-region tracking score",
     "min(frac_lin, frac_ang)")
ax.text(0.99, 0.06,
        "The STANDING run (v4) outscores the WALKING run (v6) on the very\n"
        "metric every curriculum gate used before v6 — why 4 runs were fooled.",
        transform=ax.transAxes, ha="right", va="bottom", fontsize=8.5, color=INK2)
plt.show()


In [ ]:
fig, ax = plt.subplots(figsize=(9.5, 3.8))
if "v6" in runs:
    draw(ax, [(*runs["v6"]["progress_lin"], "linear", MET_COLOR[0]),
              (*runs["v6"]["progress_ang"], "angular", MET_COLOR[1])],
         "Proof of walking — velocity-progress income (v6)",
         "episode reward income")
ax.text(0.99, 0.06, "Pays achieved/commanded velocity ratio per step;\n"
        "standing earns exactly zero. Climbing all run = real locomotion.",
        transform=ax.transAxes, ha="right", va="bottom", fontsize=8.5, color=INK2)
plt.show()


In [ ]:
fig, ax = plt.subplots(figsize=(9.5, 3.8))
draw(ax, run_series("std"), "Exploration — mean action std", "std")
ax.text(0.99, 0.97,
        "v4 annealed to 0.04 and exploited standing; v6 bottomed at 0.08 then\n"
        "re-expanded (entropy coef 0.001 -> the re-expansion is gradient-driven).",
        transform=ax.transAxes, ha="right", va="top", fontsize=8.5, color=INK2)
plt.show()


In [ ]:
fig, ax = plt.subplots(figsize=(9.5, 3.8))
if "v6" in runs:
    draw(ax, [(*runs["v6"]["err_vel_xy"], "xy (m/s)", MET_COLOR[0]),
              (*runs["v6"]["err_vel_yaw"], "yaw (rad/s)", MET_COLOR[1])],
         "Velocity error (v6) vs flat-walking baseline", "mean |error|")
for name, key in (("v21a xy", "err_vel_xy"), ("v21a yaw", "err_vel_yaw")):
    if key in baseline:
        ax.axhline(baseline[key], color=MUTED, lw=1.4, ls=(0, (5, 4)), alpha=0.8)
        ax.annotate(name, (1.0, baseline[key]), xycoords=("axes fraction", "data"),
                    xytext=(4, 0), textcoords="offset points",
                    fontsize=8.5, color=MUTED, va="center")
plt.show()


In [ ]:
fig, ax = plt.subplots(figsize=(9.5, 3.4))
if "v6" in runs:
    draw(ax, [(*runs["v6"]["term_illegal"], "illegal contact", MET_COLOR[0]),
              (*runs["v6"]["term_fell"], "fell over", MET_COLOR[1])],
         "Deaths per logging window (v6) — timeouts excluded", "terminations")
plt.show()


In [ ]:
# Phase table: window means of the key v6 metrics.
WINDOWS = [(0, 100), (100, 400), (400, 1000), (1000, 2000),
           (2000, 4000), (4000, 6000), (6000, None)]
COLS = ["terrain_levels", "progress_lin", "eplen", "reward", "std", "err_vel_xy"]

if "v6" in runs:
    hdr = f"{'iters':>12} | " + " | ".join(f"{c:>14}" for c in COLS)
    print(hdr)
    print("-" * len(hdr))
    for lo, hi in WINDOWS:
        row = []
        for c in COLS:
            it, v = runs["v6"][c]
            m = (it >= lo) & ((it < hi) if hi else np.ones_like(it, bool))
            row.append(f"{v[m].mean():14.3f}" if m.any() else " " * 14)
        hi_s = hi if hi else int(it[-1])
        print(f"{f'{lo}-{hi_s}':>12} | " + " | ".join(row))
